# Numerical Benchmarks

Focused notebook for timing, scaling, and low-level numerical sanity checks.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from gw_turbulence import H_pq, H_pq_decaying, H_pq_decaying_grid, LiveStatusLogger, g_decaying

OUT = Path("../outputs")
OUT.mkdir(exist_ok=True)
M, R, k0 = 0.1, 1e4, 1.0

## Timing Samples

Use this section to estimate point-evaluation cost before launching large scans.

In [ ]:
status = LiveStatusLogger(prefix="point-eval", every_seconds=0.5)
test_points = [(0.5, 0.6), (1.0, 0.3), (0.2, 1.5)]
rows = []
for p, q in test_points:
    t0 = time.time()
    stationary = H_pq(p, q, M=M, R=R, k0=k0, epsabs=5e-4, epsrel=3e-3)
    t1 = time.time()
    decaying = H_pq_decaying(
        p,
        q,
        M=M,
        R=R,
        k0=k0,
        convolution_method="trapz",
        convolution_points=48,
        integration_method="sampled",
        x_points=16,
        y_points=16,
        status=status,
    )
    t2 = time.time()
    rows.append((p, q, stationary, t1 - t0, decaying, t2 - t1))
rows

## Scaling Relations

Quick diagnostic plots for the Kolmogorov-style scaling blocks used in the derivation.

In [ ]:
C_k, epsilon = 1.6, 1e-3
ks = np.logspace(0, 3, 200)
A_k = C_k * epsilon**(2 / 3) * ks**(-11 / 3) / (4 * np.pi)
E_k = C_k * epsilon**(2 / 3) * ks**(-5 / 3)

plt.figure(figsize=(6, 4))
plt.loglog(ks, E_k, label="E_k")
plt.loglog(ks, A_k, label="A(k)")
plt.legend()
plt.tight_layout()
plt.show()

## Decaying Kernel Shape

Inspect the real and imaginary parts of the decaying temporal kernel before using it in large integrations.

In [ ]:
z = np.linspace(-20, 20, 600)
gz = g_decaying(z)

plt.figure(figsize=(6, 4))
plt.plot(z, gz.real, label="Re g")
plt.plot(z, gz.imag, label="Im g")
plt.legend()
plt.tight_layout()
plt.show()